In [16]:
import pandas as pd

In [18]:
df = pd.read_parquet("hf://datasets/sharjeelyunus/github-issues-dataset/github_issues_dataset.parquet")

In [19]:
df.drop(["id"], axis=1, inplace=True)

In [20]:
df.sample(5)

,repo,title,body,labels,priority,severity
35698,TypeScript,New numeric string type to accurately represen...,# Suggestion\r\n\r\n<!--\r\n Please fill in e...,"Suggestion,Awaiting More Feedback",medium,Major
113466,next.js,next lint fails when using ESLint v8 flat config,### Link to the code that reproduces this issu...,"Linting,linear: next",low,Minor
81942,vscode,Word wrap indicator,<!-- ⚠️⚠️ Do Not Delete This! feature_request_...,"feature-request,editor-wrapping",low,Minor
51428,react,Bug: hydration mismatch in top component does ...,"We render some markup on the server, but on th...","Type: Bug,Status: Unconfirmed",low,Critical
105279,go,proposal: net/http/cookiejar: add Jar.Clear,"### Proposal Details\r\n\r\nCurrently, since a...",Proposal,low,Major


In [21]:
df["repo"].value_counts()

repo
pytorch                  14451
flutter                  13130
godot                    11207
rust                      9925
go                        9094
                         ...  
d3                           3
clean-code-javascript        2
fucking-algorithm            2
awesome                      1
awesome-go                   1
Name: count, Length: 68, dtype: int64

In [22]:
df = df.apply(lambda col: col.str.lower())

In [23]:
df['severity'].value_counts()

severity
critical    66491
minor       29769
major       17813
Name: count, dtype: int64

In [24]:
df["text"] = df["repo"].fillna(" ") + " " + df["title"].fillna(" ") + " " + df["body"].fillna(" ") + " " + df["labels"].fillna(" ") + " " + df["priority"].fillna(" ")
df.drop(['repo', 'title', 'body', 'labels', 'priority'], inplace=True, axis=1) 

In [31]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english')

text_vec = vectorizer.fit_transform(df['text'])

vectorizer.get_feature_names_out()

array(['00', '000', '0000', ..., '𝟏𝟕', '𞥃a', '𠜎𠝹𠱓𠱸𠲖𠳏𠳕𠴕𠵼𠵿𠸎'],
      shape=(964923,), dtype=object)

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(text_vec, df["severity"], test_size=0.2, random_state=42)

logreg = LogisticRegression(max_iter=100)

logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)

print(accuracy_score(y_test, y_pred))

0.8170501862809555


c:\Users\arsal\Desktop\support-ticket-triage-system\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [33]:
from sklearn.svm import LinearSVC

lsvc = LinearSVC()
lsvc.fit(X_train, y_train)
y_pred_lsvc = lsvc.predict(X_test)
print(accuracy_score(y_test, y_pred_lsvc))

0.7899627438088976


c:\Users\arsal\Desktop\support-ticket-triage-system\venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
